# Frame QA — `02-extracted-frames`

Manual review companion to `tools/frame_qa.py`. The checks live in that module, so this
notebook and the CLI can never drift apart.

- **Batch / scripted:** `pixi run check` — every sequence, exit 1 if anything is blocked.
- **This notebook:** pick one sequence, read its findings, then scrub the frames in napari.

Launch with `pixi run qa`, or open in VS Code and pick the pixi `default` kernel.

Read-only: nothing here writes into `assets/`.

In [ ]:
# --- Setup --------------------------------------------------------------

import sys
from importlib import reload
from pathlib import Path

import numpy as np

TOOLS = Path.cwd() if (Path.cwd() / "frame_qa.py").exists() else Path.cwd() / "tools"
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

import frame_qa as qa
reload(qa)          # pick up edits to frame_qa.py without restarting the kernel

ROOT = qa.find_root(TOOLS)
FOUND = qa.discover(ROOT)

print(f"repo root : {ROOT}")
print(f"canvas    : {qa.CANVAS[0]}x{qa.CANVAS[1]}")
print(f"found     : {len(FOUND)} sequence(s) with 02-extracted-frames")

## Pick a sequence

Every detected sequence is checked, then listed with its verdict. Choose one from the
dropdown and **re-run the cells below it** — the selection is read when they execute, not
when you change it.

In [ ]:
# --- Check everything, then choose one ----------------------------------

REPORTS = {(seq, rev): qa.run(seq, rev, root=ROOT) for seq, rev in FOUND}

MARK = {"PASS": "ok", "PASS WITH WARNINGS": "warn", "BLOCKED": "BLOCK"}

print(f"{"":6} {"sequence":<24} {"frames":>7} {"crops":>7}   verdict")
print("-" * 72)
for key, rep in REPORTS.items():
    res = rep.resolved
    print(f"{MARK[rep.verdict]:6} {res.label:<24} {len(res.frames):>7} "
          f"{len(res.crops):>7}   {rep.verdict}")

options = [
    (f"{rep.resolved.label}  —  {rep.verdict}"
     f" ({len(rep.blocks)} block, {len(rep.warns)} warn)", key)
    for key, rep in REPORTS.items()
]

print()
try:
    import ipywidgets as widgets
    from IPython.display import display

    PICK = widgets.Dropdown(
        options=options,
        description="Assessing:",
        layout=widgets.Layout(width="640px"),
        style={"description_width": "90px"},
    )
    display(PICK)
    print("Change the dropdown, then re-run the cells below.")
except ImportError:
    PICK = None
    print("ipywidgets not available — set SELECTED by hand in the next cell:")
    for _, key in options:
        print(f"   {key}")

In [ ]:
# --- The selected sequence ----------------------------------------------
# With no widget, set this by hand, e.g. SELECTED = ("Layla-strolling", None)

SELECTED = PICK.value if PICK is not None else FOUND[0]

rep = qa.run(*SELECTED, root=ROOT)   # fresh, so on-disk edits are picked up
res = rep.resolved

print(f"=== {res.label} ===")
print(f"  package: {res.folder.parent.relative_to(ROOT)}")
print(f"  stem   : {res.stem}   (of {res.stems_seen})")
print(f"  layout : {res.layout}"
      f"{f' — frames from {res.frames_dir.name}' if res.frames_dir else ''}")
print(f"  frames : {len(res.frames)} at {qa.CANVAS[0]}x{qa.CANVAS[1]}")

if res.crops:
    from PIL import Image
    print(f"  crops  : {len(res.crops)} at {sorted({Image.open(p).size for p in res.crops})}")
if rep.lift:
    print(f"  lift   : baseline y={rep.baseline}, per-frame {rep.lift}")
if res.foreign:
    print(f"  foreign: {res.foreign}")
if res.other:
    print(f"  other  : {res.other}")

print()
for finding in rep.findings:
    print(f"  {finding}")
if not rep.findings:
    print("  no findings")

print()
print(f"{rep.verdict} — {len(rep.blocks)} blocker(s), {len(rep.warns)} warning(s).")
if rep.blocks:
    print("Do not generate 03-qa-reference or the GIF yet.")

## Visual check in napari

The automated checks catch registration and packaging faults. They cannot judge whether the
*motion* reads correctly — scrub the stack for that.

This runs whatever the sequence has. If it has been normalized you get the registered
frames and every overlay is meaningful. If it has **not**, you get the tight crops padded
onto a common canvas so they can still be viewed — useful for judging cutout quality and
pose, useless for judging alignment, so the baseline overlay is suppressed in that mode and
the cell says so loudly.

| Layer | What to look for |
| --- | --- |
| `frames` / `crops (padded)` | the sequence; use the slider as a scrubber |
| `alpha` | cutout quality — halos, holes, leftover background |
| `delta` | frame-to-frame change; a near-black slice means a wasted frame |
| `baseline` | grounded frames should sit on this line *(registered only)* |
| `bbox` | per-frame content bounds; watch for horizontal drift |

napari opens in its own window (PyQt6), not inline. Press play beside the frame slider to
loop the sequence.

In [ ]:
# --- napari ---------------------------------------------------------------
# Views the registered frames when they exist, and falls back to the tight crops
# when they do not, so a sequence can be eyeballed before it is normalized.

import napari

stack, indices, REGISTERED = qa.review_stack(res)

if stack.shape[0] == 0:
    raise SystemExit(f"{res.label} has no frames or crops in {res.folder.name}")

boxes = [qa.content_bbox(f) for f in stack]
h, w = stack.shape[1:3]

if not REGISTERED:
    print("!" * 72)
    print(f"UNREGISTERED — showing {len(indices)} tight crop(s), not normalized frames.")
    print(f"Each crop is centred and sat on the bottom of a {w}x{h} pad purely so they")
    print("can be stacked. That padding is NOT the pipeline baseline.")
    print("Judge cutout quality, pose and motion. Do NOT judge drift, lift or alignment.")
    print("!" * 72)
    print()

delta = np.zeros_like(stack[..., :3])
delta[1:] = np.abs(stack[1:, ..., :3].astype(np.int16)
                   - stack[:-1, ..., :3].astype(np.int16)).astype(np.uint8)

title = f"{res.label} — " + ("02-extracted-frames" if REGISTERED else "UNREGISTERED CROPS")
viewer = napari.Viewer(title=title)
viewer.add_image(stack, name="frames" if REGISTERED else "crops (padded)", rgb=True)
viewer.add_image(stack[..., 3], name="alpha", colormap="gray", visible=False)
viewer.add_image(delta, name="delta", rgb=True, visible=False)

# A baseline line is only meaningful once frames share a registration canvas.
if REGISTERED and rep.baseline is not None:
    viewer.add_shapes(
        [np.array([[rep.baseline, 0.0], [rep.baseline, w - 1.0]])],
        shape_type="line", edge_color="#ff3b6b", edge_width=1, name="baseline",
    )

rects = [
    np.array([[i, b[1], b[0]], [i, b[1], b[2]], [i, b[3], b[2]], [i, b[3], b[0]]], dtype=float)
    for i, b in enumerate(boxes) if b
]
if rects:
    viewer.add_shapes(
        rects, shape_type="polygon", face_color="transparent",
        edge_color="#4da3ff", edge_width=1, name="bbox", visible=False,
    )

if rep.spec and rep.spec.get("timing_ms"):
    timing = rep.spec["timing_ms"]
    mean_ms = sum(timing) / len(timing)
    print(f"spec timing: {timing}")
    print(f"mean {mean_ms:.0f}ms/frame -> set playback to {1000 / mean_ms:.1f} fps")
    print("(napari plays at a constant rate; uneven timing only reads right in the GIF)")
    print()

kind = "frames" if REGISTERED else "padded crops"
print(f"{len(indices)} {kind} loaded. Scrub the slider at the bottom of the viewer.")

## When this passes

Only then generate `03-qa-reference/` (labelled contact sheet + preview GIF) and finalise
`04-integration-notes/`. Then update the **Sequence registry** table in `CLAUDE.md`.

If a sequence needs re-generating, it goes in a new `revision-NN/` — never overwrite.

To change a threshold, edit the constants at the top of `tools/frame_qa.py` and re-run the
setup cell; `reload()` picks the change up without restarting the kernel.